# 02 — Responses API (Layer 1, Part B)

**Goal.** Re-do the work from notebook #1 on the **Responses API** — the surface Microsoft and OpenAI are steering all new work toward — and pick up three capabilities Chat Completions doesn't give you:

1. **Server-side conversation state** via `previous_response_id` (no more manual message list).
2. **Built-in tools** — we'll wire `web_search` so the model can pull live market context.
3. **File input** — feed `gpt-5.4` a PDF research note alongside the adviser's question.

We'll also flip `reasoning_effort` from `low` → `medium` on one harder question so you can see the latency / quality trade-off.

> Same plumbing as notebooks #0–#1: `DefaultAzureCredential` against `r2d2-foundry-001.openai.azure.com/openai/v1/`, deployment `gpt-5.4`. The Responses API lives at the same v1 GA endpoint — just `client.responses.*` instead of `client.chat.completions.*`.

## Step 1 — Setup

Optional install of `reportlab` (used in Step 5 to mint a tiny one-page PDF on the fly so the notebook is self-contained — no external files needed).

In [ ]:
 #%pip install --quiet reportlab

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sys, pathlib, json
from dataclasses import asdict

if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))

from _common.env import load_env
from _common.clients import get_credential, get_openai_client
from _common.scenario import CLIENT_PROFILE, portfolio_summary, WATCHLIST

cfg = load_env()
credential = get_credential()
client = get_openai_client(cfg, credential=credential)

client_ctx = {
    "profile": asdict(CLIENT_PROFILE),
    "portfolio_summary": portfolio_summary(),
    "watchlist": WATCHLIST,
}

INSTRUCTIONS = (
    "You are the Cobalt Advisory Co-Pilot. Be concise and factual. Never give individualized\n"
    "investment advice — surface trade-offs the adviser can discuss with the client. Cite figures back\n"
    "to the client_context JSON. Say 'I don't have that data' when something isn't in context.\n\n"
    f"client_context = {json.dumps(client_ctx, default=str)}"
)

print("Deployment:", cfg.model_deployment, "| reasoning_effort:", cfg.reasoning_effort)

Deployment: gpt-5.4 | reasoning_effort: low


## Step 2 — Single-turn (the Responses API equivalent)

Three small API shape differences vs. Chat Completions:

| Chat Completions | Responses |
|---|---|
| `client.chat.completions.create` | `client.responses.create` |
| `messages=[{system}, {user}, ...]` | `instructions=...` + `input="..."` (string or message list) |
| `resp.choices[0].message.content` | `resp.output_text` (convenience accessor) |

Server-issues `resp.id` is the key to multi-turn state in Step 3.

In [2]:
resp = client.responses.create(
    model=cfg.model_deployment,
    reasoning={"effort": cfg.reasoning_effort},
    instructions=INSTRUCTIONS,
    input=(
        "Avery is asking why we still hold so much NVDA. Give me a 4-bullet talking-points list "
        "covering: concentration vs. her 10% cap, unrealized gain implications, ESG/risk fit, "
        "and one diversification option."
    ),
)

print("response_id:", resp.id)
print("\n--- Reply ---\n")
print(resp.output_text)
print("\n--- Usage ---")
print("input:", resp.usage.input_tokens, "| output:", resp.usage.output_tokens, "| total:", resp.usage.total_tokens)

response_id: resp_042c060698b3484a006a0db73b704c81909fcaf851162b4b35

--- Reply ---

- **Concentration vs. Avery’s 10% cap:** Avery’s stated constraint is to **avoid single-stock concentration above 10% of the portfolio** (`constraints`). I **don’t have the current NVDA position size** in the context, so I can’t confirm whether it is above that cap. If it is near or over 10% of her **$1.42M portfolio** (`portfolio_summary.total_market_value_usd`), that would be the clearest reason to discuss trimming.

- **Unrealized gain implications:** The household portfolio has about **$362,042.50 of unrealized gains** (`portfolio_summary.approx_unrealized_gain_usd`). I **don’t have the NVDA-specific embedded gain**, but if NVDA is a large winner, selling all at once could create a meaningful tax cost. That’s the main trade-off: reducing concentration risk versus realizing gains now, especially given her **tax-aware preference to harvest losses and use ETFs in taxable accounts** (`constraints`).

-

## Step 3 — Multi-turn with server-side state

Pass `previous_response_id=<id from the last call>` and the model picks up the full prior context — instructions, inputs, reasoning, the lot. No more bookkeeping on the client.

Same three-turn conversation as notebook #1, Step 4 — compare the code shrinkage.

In [3]:
turn1 = client.responses.create(
    model=cfg.model_deployment,
    reasoning={"effort": cfg.reasoning_effort},
    instructions=INSTRUCTIONS,
    input="What is Avery's current cash position and how does it compare to her stated 6-month emergency reserve goal?",
)
print("Turn 1:\n", turn1.output_text)

turn2 = client.responses.create(
    model=cfg.model_deployment,
    reasoning={"effort": cfg.reasoning_effort},
    previous_response_id=turn1.id,
    input="Given that, what's a reasonable upper bound on how much of CASH I could redeploy this quarter without breaking her constraints?",
)
print("\nTurn 2:\n", turn2.output_text)

turn3 = client.responses.create(
    model=cfg.model_deployment,
    reasoning={"effort": cfg.reasoning_effort},
    previous_response_id=turn2.id,
    input="Summarize this conversation as three bullet points for my CRM note.",
)
print("\nTurn 3:\n", turn3.output_text)

Turn 1:
 Avery’s current brokerage cash position is about **$109,340**, based on **7.7% cash** of a **$1,420,000** portfolio (`portfolio_summary.total_market_value_usd` and `allocation_by_asset_class_pct.cash`).

On the emergency reserve goal: Avery’s stated goal is to **“Maintain 6-month emergency reserve outside brokerage”** (`profile.goals`). I **don’t have the monthly spending data** needed to calculate what 6 months of expenses should equal, and I also **don’t have a confirmed dollar amount for cash held outside the brokerage**.

So the comparison is:

- **Brokerage cash:** ~**$109,340**
- **Required 6-month reserve:** **I don’t have that data**
- **Outside-brokerage reserve currently held:** **I don’t have that data**

A useful follow-up for the adviser would be to confirm Avery’s monthly core spending and how much of the **$210,000 in liquid assets** (`profile.liquid_assets_usd`) is held outside the brokerage as true emergency cash.

Turn 2:
 A **reasonable upper bound** is **ab

## Step 4 — Built-in `web_search` tool

Chat Completions makes you wire your own function-calling loop for web search. The Responses API ships `web_search` as a hosted tool — pass `tools=[{"type": "web_search"}]` and the model decides when to invoke it. You get the synthesized answer in `output_text` and the tool-call/results in the `output` array for auditing.

We'll ask a question that *requires* fresh information.

In [4]:
resp = client.responses.create(
    model=cfg.model_deployment,
    reasoning={"effort": "medium"},  # bumped — research-style query benefits from more reasoning
    instructions=INSTRUCTIONS,
    tools=[{"type": "web_search"}],
    input=(
        "Search the web for the most recent NVIDIA quarterly earnings release and summarize three "
        "points that Avery (currently overweight NVDA) should be aware of. End with a one-line citation list."
    ),
)

print("--- Synthesized answer ---\n")
print(resp.output_text)

print("\n--- Output trace (tool calls + final message) ---")
for item in resp.output:
    print(f"  - type={item.type}")

--- Synthesized answer ---

As of my search on **May 20, 2026**, the most recent **publicly posted** NVIDIA quarterly earnings release I could verify is the **Q4 / Fiscal 2026** release dated **February 25, 2026**. NVIDIA had its **Q1 FY27** results call scheduled for **May 20, 2026**, but the official financial reports page still showed Q4/FY26 as the latest posted earnings release when I checked. ([investor.nvidia.com](https://investor.nvidia.com/financial-info/financial-reports/default.aspx))

Given your note that Avery is **overweight NVDA**, and Avery’s firm-held portfolio is **$1.42M** with a stated constraint to **avoid single-stock concentration above 10%** (client context), here are **three points to discuss**:

1. **The growth was still exceptional, but expectations are now very high.** NVIDIA reported **Q4 revenue of $68.1B**, up **20% sequentially** and **73% year over year**; **GAAP EPS was $1.76**, up **98% year over year**. The trade-off for an overweight holder is that 

## Step 5 — PDF file input

Real adviser workflows pull in research notes, fund factsheets, prospectuses — all PDFs. The Responses API takes file inputs natively: upload via `client.files.create(purpose="assistants")`, then reference the `file_id` in the `input` content array.

We mint a tiny one-page research note on the fly so the notebook stays self-contained.

In [5]:
from reportlab.lib.pagesizes import LETTER
from reportlab.pdfgen import canvas
import io, textwrap

buf = io.BytesIO()
c = canvas.Canvas(buf, pagesize=LETTER)
c.setFont("Helvetica-Bold", 14); c.drawString(72, 740, "Cobalt Research — Semiconductors Brief (Internal Use)")
c.setFont("Helvetica", 10);     c.drawString(72, 722, "Date: 2026-05-12   Analyst: J. Park   Sector: Semiconductors")

body = textwrap.dedent('''
    Summary: We are downgrading the US semis basket from OVERWEIGHT to MARKET-WEIGHT.
    Three drivers:
      (1) AI-accelerator pricing power is normalizing as TSM 3nm capacity ramps in H2.
      (2) Channel inventory for high-end GPUs has crept above 12 weeks at top-3 OEMs.
      (3) Power and water permitting in AZ/TX adds 4-6 months to fab expansion timelines.

    Stock view (NVDA): TRIM positions above 8% portfolio weight. Long-term thesis intact;
    near-term setup favors taking some gains and rebalancing into broad-tech ETFs (e.g. QQQM)
    or diversified semis (e.g. SOXX/SMH) to maintain factor exposure without single-name risk.

    Risk to view: Materially stronger-than-expected sovereign AI build-outs (UAE, Saudi, EU)
    in H2 would re-tighten H100/H200/B100 supply and invalidate the inventory signal.
''').strip().splitlines()

y = 700
for line in body:
    c.drawString(72, y, line); y -= 14
c.showPage(); c.save()
pdf_bytes = buf.getvalue()

pdf_path = pathlib.Path("semis_brief_2026-05-12.pdf")
pdf_path.write_bytes(pdf_bytes)
print(f"Wrote {pdf_path} ({len(pdf_bytes):,} bytes)")

Wrote semis_brief_2026-05-12.pdf (2,328 bytes)


In [6]:
# Upload to the OpenAI Files endpoint, then reference by file_id in the input array.
with open(pdf_path, "rb") as f:
    uploaded = client.files.create(file=f, purpose="assistants")
print("Uploaded file_id:", uploaded.id)

resp = client.responses.create(
    model=cfg.model_deployment,
    reasoning={"effort": "medium"},
    instructions=INSTRUCTIONS,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_file", "file_id": uploaded.id},
                {
                    "type": "input_text",
                    "text": (
                        "Read the attached internal research brief. Map its recommendations to Avery's "
                        "current portfolio (especially NVDA weight) and give me a 3-bullet adviser-ready "
                        "recap: (1) what the brief says, (2) how it applies to Avery, (3) a concrete next step."
                    ),
                },
            ],
        }
    ],
)

print("\n--- Adviser recap ---\n")
print(resp.output_text)

Uploaded file_id: file-57a99c7a8d354702ad46c2197a28288e


BadRequestError: Error code: 400 - {'error': {'message': "Invalid 'input[0].content[0].file_id': 'file-57a99c7a8d354702ad46c2197a28288e'. Expected an ID that begins with 'assistant'.", 'type': 'invalid_request_error', 'param': 'input[0].content[0].file_id', 'code': 'invalid_value'}}

## Step 6 — Structured output with a JSON schema

Notebook #1 used `response_format={"type":"json_object"}` (free-form JSON, shape only hinted in the prompt). The Responses API supports proper **JSON-schema-validated** output via `text.format = {"type": "json_schema", "name": ..., "schema": {...}, "strict": true}` — the server rejects any reply that doesn't match the schema, so your downstream code can `json.loads` without defensive parsing.

In [ ]:
concentration_schema = {
    "type": "object",
    "additionalProperties": False,
    "required": ["summary", "concentration_flags", "recommended_next_actions"],
    "properties": {
        "summary": {"type": "string"},
        "concentration_flags": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "required": ["symbol", "pct_of_portfolio", "breaches_10pct_cap"],
                "properties": {
                    "symbol": {"type": "string"},
                    "pct_of_portfolio": {"type": "number"},
                    "breaches_10pct_cap": {"type": "boolean"},
                },
            },
        },
        "recommended_next_actions": {"type": "array", "items": {"type": "string"}},
    },
}

resp = client.responses.create(
    model=cfg.model_deployment,
    reasoning={"effort": cfg.reasoning_effort},
    instructions=INSTRUCTIONS,
    text={
        "format": {
            "type": "json_schema",
            "name": "ConcentrationReport",
            "schema": concentration_schema,
            "strict": True,
        }
    },
    input=(
        "From the client_context holdings, list every position whose market value exceeds 10% of total "
        "portfolio market value as a concentration flag, then propose 2-3 next actions for the adviser."
    ),
)

parsed = json.loads(resp.output_text)
print(json.dumps(parsed, indent=2))

## Recap & what's next

Three things just got materially easier:

- **Conversation state** — `previous_response_id` instead of a manually-curated message list.
- **Tools** — `web_search` (and others: `file_search`, `code_interpreter`, `computer_use`) are one-line wires; no client-side function-calling loop.
- **File input** — upload once, reference by `file_id`; the model parses the PDF for you.

But you'll notice we still hard-coded `client_ctx` into `instructions`. For a real adviser desk with thousands of clients and dozens of research PDFs, that doesn't scale. That's what **notebook #3 — Foundry Agent Service** solves: a *hosted, named, persistent agent* with a `file_search` vector store, threaded conversations, and managed identity wiring so multiple sessions / users / surfaces can talk to the same configured agent.